In [0]:
# --- 1. Global Config (Match your paths.py) ---
CATALOG = "lending_risk"
SCHEMAS = ["landing", "bronze", "silver", "gold"]

# --- 2. Namespace Initialization ---
print(f"Creating Catalog: {CATALOG}")
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")

for schema in SCHEMAS:
    print(f"Creating Schema: {schema}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

# --- 3. Volume Initialization (The Landing & Checkpoint Zones) ---
print(f"Creating Volume: {CATALOG}.landing.vol_landing")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.landing.vol_landing")

print(f"Creating Volume: {CATALOG}.bronze.vol_bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.bronze.vol_bronze")

# --- 4. Directory Structure Creation (ADLS/Volume folders) ---
# Directories inside Volumes must be created via dbutils.fs
folders = ["customers", "loans", "defaulters", "repayments"]
LANDING_PATH = f"/Volumes/{CATALOG}/landing/vol_landing"

for folder in folders:
    folder_path = f"{LANDING_PATH}/{folder}"
    print(f"Ensuring Directory: {folder_path}")
    dbutils.fs.mkdirs(folder_path)

# --- 5. Table Initialization ---
# We use dbutils.notebook.run to execute the layer-specific DDL scripts
print("Building Bronze Layer...")
dbutils.notebook.run("01_create_tables_bronze", 600)

print("Building Silver Layer...")
dbutils.notebook.run("02_create_tables_silver", 600)

print("Building Gold Layer...")
dbutils.notebook.run("03_create_tables_gold", 600)

print("--- LENDING RISK ENVIRONMENT READY ---")